<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E3_DBSCAN_vs_KMeans_Moons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · DBSCAN vs K-Means en Moons - Análisis cluster

## Introducción

K-Means agrupa alrededor de centros, así que funciona bien con grupos **compactos y
redondeados**, pero falla con formas alargadas o curvas. **DBSCAN** agrupa por **densidad**:
une zonas con muchos puntos juntos y marca como **ruido** los puntos aislados. No necesita fijar
K y encuentra formas irregulares.

Lo vemos con el caso de libro: dos **lunas** (medias lunas entrelazadas). Aquí sí generamos los
datos a propósito (`make_moons`), porque es la forma perfecta para ver la diferencia.

## Objetivos del ejercicio

- Ver que K-Means **parte mal** las formas no compactas.
- Aplicar **DBSCAN** y entender sus dos parámetros: `eps` (ε) y `min_samples` (minPts).
- Variar `eps`/`minPts`, identificar los **puntos de ruido** y comparar con K-Means.

## Descripción del dataset (Moons)

`make_moons` genera dos medias lunas entrelazadas con algo de ruido. No son grupos compactos,
así que es el ejemplo ideal para comparar un método basado en centros (K-Means) con uno basado
en densidad (DBSCAN).

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN

### 2. Generar las lunas y escalar

In [ ]:
X, _ = make_moons(n_samples=500, noise=0.10, random_state=0)
X = StandardScaler().fit_transform(X)
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=12, color="#34495e")
plt.title("Datos: dos lunas (sin etiqueta)")
plt.tight_layout(); plt.show()

### 3. K-Means: parte las lunas por la mitad

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=km, cmap="coolwarm", s=12)
plt.title("K-Means (K=2): corta en recto, no respeta las lunas")
plt.tight_layout(); plt.show()

### 4. DBSCAN: agrupa por densidad

- `eps` (ε): radio de vecindad alrededor de cada punto.
- `min_samples` (minPts): cuántos vecinos hacen falta para considerar una zona "densa".

Los puntos que no entran en ninguna zona densa se marcan como **ruido** (etiqueta -1).

In [ ]:
db = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
n_clusters = len(set(db) - {-1})
n_ruido = int((db == -1).sum())
print(f"DBSCAN (eps=0.2, minPts=5) -> {n_clusters} clusters y {n_ruido} puntos de ruido")

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=db, cmap="coolwarm", s=12)
plt.title(f"DBSCAN: {n_clusters} clusters, {n_ruido} puntos de ruido (-1)")
plt.tight_layout(); plt.show()

### 5. Variar eps y minPts

In [ ]:
for eps in [0.15, 0.20, 0.35]:
    for mp in [5, 10]:
        lab = DBSCAN(eps=eps, min_samples=mp).fit_predict(X)
        nc = len(set(lab) - {-1})
        nr = int((lab == -1).sum())
        print(f"eps={eps:<4} minPts={mp:<3} -> {nc} clusters, {nr} puntos de ruido")
print("\nDBSCAN es muy sensible a eps: hay que ajustarlo (y validar con criterio).")

### Reflexión

1. ¿Por qué K-Means no puede separar bien las dos lunas?
2. ¿Qué papel juega `eps`? ¿Y `min_samples`?
3. ¿Qué pasa con el número de clusters y de ruido cuando subes mucho `eps`?
4. ¿En qué casos reales preferirías DBSCAN frente a K-Means?